# NOTEBOOK 4: MODEL 2 - XGBOOST + ENSEMBLE STRATEGY

**PURPOSE:** Train XGBoost, create ensemble with Random Forest, business segmentation

**INPUT:** X_train.csv, X_test.csv, y_train.csv, y_test.csv, rf_model.pkl

**OUTPUT:** xgb_model.pkl, ensemble_predictions.csv, business_segmentation.csv

In [109]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE

# Load data and trained Random Forest model
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

with open('rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

print(f"Data and RF model loaded")

Data and RF model loaded


## 2. APPLY SMOTE TO TRAINING DATA

In [110]:
smote = SMOTE(k_neighbors=5, random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"✓ SMOTE applied to training data")

✓ SMOTE applied to training data


## 3. CONFIGURE XGBOOST HYPERPARAMETERS

In [111]:
xgb_model = xgb.XGBClassifier(
    n_estimators=100,              # 100 boosting rounds
    learning_rate=0.1,             # Shrinkage rate (eta)
    max_depth=6,                   # Tree depth
    min_child_weight=1,            # Minimum leaf weight
    subsample=0.8,                 # Use 80% of samples per iteration
    colsample_bytree=0.8,          # Use 80% of features per tree
    gamma=0,                       # No minimum loss reduction
    reg_lambda=1.0,                    # L2 regularization
    reg_alpha=0,                       # No L1 regularization
    objective='binary:logistic',   # Binary classification
    eval_metric='logloss',         # Evaluation metric
    random_state=42,
    n_jobs=-1                      # Use all cores
)

print("✓ XGBoost configured")

✓ XGBoost configured


## 4. TRAIN XGBOOST MODEL

In [112]:
xgb_model.fit(X_train_balanced, y_train_balanced, verbose=0)
print("✓ XGBoost trained")

✓ XGBoost trained


## 5. GENERATE PREDICTIONS FROM BOTH MODELS

In [113]:
# Random Forest predictions
y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]

# XGBoost predictions
y_pred_xgb = xgb_model.predict(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print(f"✓ Predictions generated from both models")

✓ Predictions generated from both models


## 6. CREATE ENSEMBLE PREDICTIONS (AVERAGE PROBABILITY)

In [114]:
# Ensemble = (RF_Probability + XGB_Probability) / 2
y_pred_proba_ensemble = (y_pred_proba_rf + y_pred_proba_xgb) / 2
y_pred_ensemble = (y_pred_proba_ensemble >= 0.5).astype(int)

print(f"✓ Ensemble predictions created")

✓ Ensemble predictions created


## 7. SAVE ALL OUTPUTS

In [115]:
# Save XGBoost model
with open('xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

# Save ensemble predictions
ensemble_predictions = pd.DataFrame({
    'RF_Probability': y_pred_proba_rf,
    'XGB_Probability': y_pred_proba_xgb,
    'Ensemble_Probability': y_pred_proba_ensemble,
    'Ensemble_Prediction': y_pred_ensemble,
    'Actual': y_test
})

# Save ensemble predictions
ensemble_predictions.to_csv('ensemble_predictions.csv', index=False)